# AI Essentials with Python

This notebook demonstrates how a large language model (LLM) use a tool call.

We send a single question to a model running locally on your machine via [Ollama](https://ollama.com). The model calls then the tool go get the correct answer.

## Prerequisites

Before running this notebook you have to:

- Ensure [01_call_llm.ipynb](01_call_llm.ipynb) is runnable
- Change the `MODEL` variable to the model you have pulled locally

In [ ]:
MODEL = "llama3.2"

## Install the `ollama` Python package

Run the cell below to install the official Ollama client for Python.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ollama"])

## Define a tool
We create a simple tool that counts how many times a character appears in a word.

In [ ]:
def count_character(word: str, character: str) -> int:
    return word.lower().count(character.lower())

## Describe the tool for the LLM
The LLM must know when and how to call the tool. So we describe in the tool schema:
- The tool's name and a description of what it does
- What input parameters are expected
    - The parameters are a json object with
        - the word of type string
        - the character of type string
        - both parameters are required

In [ ]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'count_character_in_word',
        'description': 'Use this tool if you need to count exactly how many times a single character appears in a word',
        'parameters': {
            'type': 'object',
            'properties': {
                'word': {
                    'type': 'string',
                    'description': 'The word to search the character in',
                },
                'character': {
                    'type': 'string',
                    'description': 'The single character to count in the word',
                },
            },
            'required': ['word', 'character'],
        },
    },
}]

## Call the model with the tool
We send a question to the AI and let it use the tool to get the correct answer.

In [ ]:
import ollama

QUESTION = "How many r's are in the word strawberry?"

messages = [{'role': 'user', 'content': QUESTION}]
print(f"Question: {QUESTION}")

while True:
    response = ollama.chat(model=MODEL, messages=messages, tools=tools)

    if response.message.tool_calls:
        # see https://docs.ollama.com/capabilities/tool-calling
        messages.append({
            'role': 'assistant',
            'tool_calls': [
                {
                    'type': 'function',
                    'function': {
                        'index': idx,
                        'name': tool.function.name,
                        'arguments': tool.function.arguments,
                    }
                }
                for idx, tool in enumerate(response.message.tool_calls)
            ]
        })

        for tool in response.message.tool_calls:
            if tool.function.name == 'count_character_in_word':
                print(f"Tool call: {tool.function.name} with arguments {tool.function.arguments}: ")

                result = count_character(**tool.function.arguments)
                print(f"Tool call: {tool.function.name} result: {result}")
                # see https://docs.ollama.com/capabilities/tool-calling
                messages.append({
                    'role': 'tool',
                    'tool_name': tool.function.name,
                    'content': str(result),
                })
            else:
                raise ValueError(f"Unknown tool called: {tool.function.name}")
                
    if response.message.content:
        print(f"Answer ({MODEL}): {response.message.content}")
        break